# Validation of point_layer template_title using point observations from point_obs_source

In [ ]:
variable = "point_variable"

In [ ]:
chunk_start

In [ ]:
layer = "point_layer"
point_source = "point_obs_source"
import glob
import dill
from oceanval import plotnine_figures, stratification, unit_display
from oceanval.tidiers import fix_unit
folder = f"data_dir_value/oceanval_matchups/point/{layer}/{variable}/{point_source}"
definitions = dill.load(open(glob.glob(f"{folder}/*definitions*.pkl")[0], "rb"))
vv_name = definitions[variable].long_name
vv_source_raw = point_source

# the measures of each cast (oceanval.stratification): one row a cast and measure
df = pd.read_csv(glob.glob(f"{folder}/*_{variable}.csv")[0])
df = df.query(
    f"lon >= {lon_lim[0]} and lon <= {lon_lim[1]} and lat >= {lat_lim[0]} and lat <= {lat_lim[1]}"
).reset_index(drop = True)
cast_keys = stratification.cast_keys(df)
casts = df.loc[:, cast_keys].drop_duplicates().reset_index(drop = True)
n_casts = len(casts)
found_measures = [x for x in stratification.MEASURES if x in set(df.measure)]

def measure_title(measure, plain=False):
    unit = stratification.UNITS[measure]
    return f"{stratification.LABELS[measure]} ({unit if plain else fix_unit(unit)})"

def years_text(df):
    if len(df) == 0:
        return ""
    first, last = int(df.year.min()), int(df.year.max())
    return f"{first}" if first == last else f"{first}-{last}"

# the summer, for the maps: June to August north of the equator, December to February south of it
summer = [6, 7, 8] if df.lat.mean() >= 0 else [12, 1, 2]
summer_name = "June to August" if summer[0] == 6 else "December to February"

In [ ]:
md(f"This page compares how stratified the water column is in the model and in {n_casts} casts of {point_source} observations ({years_text(df)}). Each cast is compared with the model's water column nearest it, on the day it was made, and each measure below is worked out the same way from the observed cast and from the model at the cast's depths:")
md(f"- **{measure_title('dt')}**: the temperature at the surface minus that at the bottom. A cast is stratified when it is over {stratification.STRATIFIED_DT} °C.\n"
   f"- **{measure_title('pea_t')}**: the energy needed to mix the water column (Simpson, 1981), from its temperature, the salinity held at {stratification.FIXED_SALINITY:g}. A cast is stratified when it is over {stratification.STRATIFIED_PEA:g} J m<sup>-3</sup>.\n"
   f"- **{measure_title('pea')}**: the same, from the temperature and the salinity.\n"
   f"- **{measure_title('mld')}**: the depth below {stratification.MLD_REFERENCE:g} m at which the temperature first differs from its value at {stratification.MLD_REFERENCE:g} m by more than {stratification.MLD_THRESHOLD} °C (de Boyer Montégut et al., 2004), or the water depth if it never does.", number = False)
md(f"The surface is the centre of the model's top cell, and the bottom the centre of its bottom cell; the observed cast is interpolated to the same depths. The potential energy anomaly is of the top {stratification.PEA_MAX_DEPTH:g} m at most. A cast is used if it has at least {stratification.MIN_SAMPLES} depths, the shallowest within {stratification.SURFACE_TOLERANCE:g} m of the surface.")

In [ ]:
# where the casts are (the detailed report only)
if not concise and wanted("strat_locations_map") and n_casts > 0:
    md("## Where the casts are")
    _fig = figures.call(plotnine_figures.sized, Of(plotnine_figures.locations_map, casts.loc[:, ["lon", "lat"]].drop_duplicates()), 600, 500)
    figures.show(i_figure)

In [ ]:
if not concise and wanted("strat_locations_map") and n_casts > 0:
    md(f"**Figure {i_figure}:** Locations of the {point_source} casts compared with the model.", gallery="Where the casts are")
    i_figure += 1

In [ ]:
# the seasonal cycle of each measure, observed and modelled (oceanval.plotnine_figures)
if wanted("strat_seasonal") and n_casts > 0:
    md("## Seasonal cycle of stratification")
    _fig = figures.call(plotnine_figures.sized, Of(plotnine_figures.stratification_cycle, df), 1000, 700)
    figures.show(i_figure)

In [ ]:
if wanted("strat_seasonal") and n_casts > 0:
    md(f"**Figure {i_figure}:** The mean of each measure over the casts of each month, observed and modelled. Each panel has its own scale.", gallery="Seasonal cycle of stratification")
    i_figure += 1

In [ ]:
# the summer's surface minus bottom temperature at each place, and the model's bias
df_summer = df.query("measure == 'dt' and month in @summer")
summer_text = f"in {summer_name}"
if len(df_summer) == 0:
    # no casts in the summer: every month's
    df_summer = df.query("measure == 'dt'")
    summer_text = "in every month (there are no casts from " + summer_name + ")"
df_summer = df_summer.groupby(["lon", "lat"], as_index = False)[["observation", "model"]].mean()
show_summer = wanted("strat_maps") and len(df_summer) > 0
if show_summer:
    md("## Stratification in summer")
    md(f"Figures {i_figure} and {i_figure + 1} show the mean surface minus bottom temperature of the casts at each place {summer_text}, in the model and the observations, and the model's bias.")
    dt_label = f"Surface minus bottom temperature ({unit_display.mathtext('°C')})"
    _fig = figures.call(plotnine_figures.sized, Of(plotnine_figures.mean_map, df_summer, dt_label), 1000, 700)
    figures.show(i_figure)

In [ ]:
if show_summer:
    md(f"**Figure {i_figure}:** Mean surface minus bottom temperature {summer_text}, in the model and the observations. The colours are capped at the 98th percentile.", gallery="Summer stratification: model and observations")
    i_figure += 1

In [ ]:
if show_summer:
    summer_bias, summer_high = plotnine_figures.capped_bias(df_summer)
    _fig = figures.call(plotnine_figures.sized, Of(plotnine_figures.bias_map, summer_bias, summer_high, f"Bias in surface minus bottom temperature ({unit_display.mathtext('°C')})"), 1000, 800)
    figures.show(i_figure)

In [ ]:
if show_summer:
    md(f"**Figure {i_figure}:** Bias in the mean surface minus bottom temperature {summer_text}: model minus observation, so red is where the model is more stratified than observed. The colour scale is capped at the 98th percentile of the absolute bias.", gallery="Summer stratification: model minus observations")
    i_figure += 1

In [ ]:
# each cast's measures, modelled against observed
if wanted("strat_scatter") and n_casts > 0:
    md("## Model versus observations")
    _fig = figures.call(plotnine_figures.sized, Of(plotnine_figures.stratification_scatter, df), 1000, 900)
    figures.show(i_figure)

In [ ]:
if wanted("strat_scatter") and n_casts > 0:
    md(f"**Figure {i_figure}:** Each cast's measures in the model against those observed, with the 1:1 line. Each panel has its own scales.", gallery="Model against observations, cast by cast")
    i_figure += 1

In [ ]:
# the statistics of each measure, and how often the model and the observations agree on
# whether a cast is stratified; written to the results whether or not they are shown
def measure_stats(part):
    return pd.Series({
        "Number of casts": len(part),
        "Observed mean": part.observation.mean(),
        "Model mean": part.model.mean(),
        "Bias": part.model.mean() - part.observation.mean(),
        "RMSD": np.sqrt(((part.model - part.observation) ** 2).mean()),
        "Correlation": part.model.corr(part.observation) if len(part) > 2 else np.nan,
    })

df_stats = pd.DataFrame(
    [measure_stats(df.query("measure == @x")).rename(measure_title(x, plain = True)) for x in found_measures]
).rename_axis("Measure").reset_index()
df_agree = stratification.agreement(df)
df_agree = df_agree.assign(
    measure = [measure_title(x, plain = True) for x in df_agree.measure],
    agreement = [f"{100 * x:.0f}%" for x in df_agree.agreement],
).rename(columns = {
    "measure": "Measure", "both stratified": "Both stratified", "observed only": "Observed only",
    "modelled only": "Modelled only", "both mixed": "Both mixed", "agreement": "Agreement",
})
ff_out = f"../../oceanval_results/{variable}_{point_source}_statistics.csv"
os.makedirs(os.path.dirname(ff_out), exist_ok = True)
df_stats.to_csv(ff_out, index = False)
df_agree.to_csv(ff_out.replace("_statistics.csv", "_agreement.csv"), index = False)
# shown with the units as the report writes them
html_titles = {measure_title(x, plain = True): measure_title(x) for x in stratification.MEASURES}
df_stats = df_stats.assign(Measure = df_stats.Measure.map(html_titles))
df_agree = df_agree.assign(Measure = df_agree.Measure.map(html_titles))
if wanted("strat_table") and n_casts > 0:
    md("## Statistics")
    df_display(df_stats)

In [ ]:
if wanted("strat_table") and n_casts > 0:
    md(f"**Table {i_table}:** The number of casts, the observed and modelled means, the bias (model minus observation), the root mean square difference (RMSD) and the correlation of each measure over the casts.", gallery="Statistics of each measure")
    i_table += 1
    if len(df_agree) > 0:
        df_display(df_agree)

In [ ]:
if wanted("strat_table") and n_casts > 0 and len(df_agree) > 0:
    md(f"**Table {i_table}:** How often the model and the observations agree on whether a cast is stratified (surface minus bottom temperature over {stratification.STRATIFIED_DT} °C, potential energy anomaly over {stratification.STRATIFIED_PEA:g} J m<sup>-3</sup>) or mixed: the number of casts stratified in both, in the observations only, in the model only, and mixed in both, and the share they agree on.", gallery="Stratified or mixed: agreement")
    i_table += 1

In [ ]:
chunk_end

In [ ]:
md(f"## Data Sources for validation of {vv_name}")
md_basic(definitions[variable].sources[vv_source_raw])

In [ ]:
if test_status:
    md("This is getting to the end!")